# 1.3 — Sequential Python versus PySpark

**Chapter 1, section 1.11** (*A First Distributed Program*), closing argument.

**The question this notebook answers:** the two versions compute the same thing, so what is
actually different about them?

Seven small algorithms, each written twice. The chapter's claim is that the difference is not
syntactic:

> The Python version specifies *how* the result is computed: visit element 0, then element 1,
> accumulating along the way. The PySpark version specifies *what* is to be computed, namely
> the combination of the elements under a given associative operation, and leaves the system
> free to choose the order, the grouping, and the machines.

Each pair below is followed by a line naming **the property that makes the PySpark version
parallelizable**, because that property — not the shorter syntax — is the thing to take away.
Section 5 is the anti-pattern: PySpark used in a way that throws the property away.

Supports **Exercise 4** (section 4 below) and **Exercise 2** on associativity.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 1, section 1.11.  The session creates and owns the context; `sc` is how the
# RDD API is reached.  No findspark, no sqlContext -- both are Spark 1.x.
import os, math, tempfile
from collections import Counter
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-1.3")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print("Spark", spark.version, "on", sc.master)

Spark 4.2.0 on local[*]


## 1. Sum of a list

In [2]:
# Python: an accumulator, mutated in a loop, one element at a time.
arr = [1, 2, 3, 4, 5]
result = 0
for i in range(len(arr)):
    result += arr[i]
print(result)

15


In [3]:
# PySpark: an associative combine.  Order and grouping are left to the system.
rdd = sc.parallelize([1, 2, 3, 4, 5])
result = rdd.reduce(lambda a, b: a + b)
print(result)

15


**The property:** addition is *associative*, so `(a+b)+c` and `a+(b+c)` are the same number.
That is exactly what lets the system split the list, sum each piece on a different machine, and
sum the partial sums. The Python loop fixes an order the answer never depended on.

## 2. Word count

In [4]:
# Python
text = "hello world hello"
word_dict = {}
for word in text.split():
    word_dict[word] = word_dict.get(word, 0) + 1
print(word_dict)

{'hello': 2, 'world': 1}


In [5]:
# PySpark
text = sc.parallelize(["hello world hello"])
result = (text.flatMap(lambda line: line.split(" "))
              .map(lambda word: (word, 1))
              .reduceByKey(lambda a, b: a + b)
              .collect())
print(dict(result))

{'hello': 2, 'world': 1}


**The property:** each line is turned into words independently of every other line, and the
per-key sums are associative. The dictionary in the Python version is *shared mutable state*,
which is precisely what cannot exist across machines.

## 3. Maximum element

In [6]:
# Python
arr = [2, 4, 1, 8, 5]
max_val = arr[0]
for i in range(1, len(arr)):
    if arr[i] > max_val:
        max_val = arr[i]
print(max_val)

8


In [7]:
# PySpark
rdd = sc.parallelize([2, 4, 1, 8, 5])
result = rdd.reduce(lambda a, b: a if a > b else b)
print(result)

8


**The property:** `max` is associative *and* commutative, so any grouping and any order give
the same answer.

## 4. Counting occurrences of a value

In [8]:
# Python
arr = [2, 3, 4, 2, 8, 2]
target = 2

result = 0
for i in range(len(arr)):
    if arr[i] == target:
        result += 1
print(result)

3


In [9]:
# PySpark
rdd = sc.parallelize([2, 3, 4, 2, 8, 2])
target = 2
result = rdd.filter(lambda x: x == target).count()
print(result)

3


**The property:** the test `x == target` looks at one element and nothing else, so every
element can be tested at the same time as every other. A predicate with that property is
called *element-wise*, and it is the easiest kind of work to distribute.

Note also that `target` is a **driver** variable. Spark serializes it and ships a *copy* to
every executor along with the lambda. Reading it this way is fine; assigning to it inside the
lambda would not be, and chapter 2 returns to that at length.

### Exercise 4

> *Rewrite the following in PySpark without a `for` loop, and explain in one sentence what
> property of your replacement makes it parallelizable.*
>
> ```python
> total = 0
> for x in arr:
>     if x % 2 == 0:
>         total += x * x
> ```

The loop does three separate things at once — select, transform, combine — and that is what
makes it hard to distribute. Separating them gives one operation per step, and each step has a
property that permits it to run in parallel.

In [10]:
arr = list(range(1, 21))

# Python, as given.
total = 0
for x in arr:
    if x % 2 == 0:
        total += x * x
print("python :", total)

# PySpark: filter (element-wise), map (element-wise), reduce (associative).
rdd = sc.parallelize(arr)
total_spark = (rdd
    .filter(lambda x: x % 2 == 0)      # element-wise: each x tested on its own
    .map(lambda x: x * x)              # element-wise: each x squared on its own
    .reduce(lambda a, b: a + b))       # associative: the system picks the grouping
print("pyspark:", total_spark)

assert total == total_spark
print("\nThe answer to the exercise's second half: filter and map look at one element at a")
print("time, so every element can be handled simultaneously; and + is associative, so the")
print("partial sums can be combined in whatever order and grouping the system chooses.")

python : 1540
pyspark: 1540

The answer to the exercise's second half: filter and map look at one element at a
time, so every element can be handled simultaneously; and + is associative, so the
partial sums can be combined in whatever order and grouping the system chooses.


## 5. The anti-pattern

The chapter says the explicit loop "must be avoided". It is worth seeing *how* a loop gets
smuggled back in, because it does not look like a mistake.

The cell below is written in PySpark. It gives the right answer. It is nevertheless the Python
version with extra steps, because `collect()` brings every record to the driver and then a
Python `for` loop walks them one at a time. The cluster did the easy part and then handed the
work back.

In [11]:
# WRONG: PySpark in name only.
rdd = sc.parallelize(range(1, 21))

values = rdd.collect()                 # <-- every record onto ONE machine, the driver
total = 0
for x in values:                       # <-- and then a sequential loop over them
    if x % 2 == 0:
        total += x * x
print("answer:", total, " (correct, and computed entirely on the driver)")

print("\nWhy this is the wrong answer even though the number is right:")
print("  1. collect() requires the whole data set to fit on one machine, which is the")
print("     premise distributed processing exists to escape.")
print("  2. The loop is sequential, so the executors sit idle while the driver works.")
print("  3. Nothing about it can be optimized: Spark saw a collect() and stopped.")
print("\nThe test is not 'did I import pyspark'.  It is 'where does the computation happen'.")

answer: 1540  (correct, and computed entirely on the driver)

Why this is the wrong answer even though the number is right:
  1. collect() requires the whole data set to fit on one machine, which is the
     premise distributed processing exists to escape.
  2. The loop is sequential, so the executors sit idle while the driver works.
  3. Nothing about it can be optimized: Spark saw a collect() and stopped.

The test is not 'did I import pyspark'.  It is 'where does the computation happen'.


## 6. Prime numbers in a range

In [12]:
# Python
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

prime_numbers = []
for i in range(2, 30):
    if is_prime(i):
        prime_numbers.append(i)
print(prime_numbers)

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


In [13]:
# PySpark.  Note that a plain named function works as well as a lambda: what Spark needs
# is that the function be a value it can serialize and ship, which is the point section 1.8
# makes about functional programming.
rdd = sc.parallelize(range(2, 30))
prime_numbers = rdd.filter(is_prime).collect()
print(prime_numbers)

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


**The property:** primality of `n` depends on `n` alone. There is no state carried from one
number to the next, so the range can be cut anywhere.

## 7. Histogram

In [14]:
# Python
arr = [1, 1, 1, 2, 2, 3]
result = {}
for i in range(len(arr)):
    result[arr[i]] = result.get(arr[i], 0) + 1
print(result)

{1: 3, 2: 2, 3: 1}


In [15]:
# PySpark.  countByValue is an ACTION: the result is an ordinary Python dict in the driver,
# so it is safe only once the data has been reduced to something small.
rdd = sc.parallelize([1, 1, 1, 2, 2, 3])
result = rdd.countByValue()
print(dict(result))

{1: 3, 2: 2, 3: 1}


**The property:** counting is a by-key sum, and sums are associative. Each partition can build
its own partial histogram and the partials can be added together.

## 8. Sum of squares

In [16]:
# Python
arr = [1, 2, 3, 4, 5]
sum_of_squares = 0
for i in range(len(arr)):
    sum_of_squares += arr[i] ** 2
print(sum_of_squares)

55


In [17]:
# PySpark: the transform and the combine separated, as in Exercise 4 above.
rdd = sc.parallelize([1, 2, 3, 4, 5])
sum_of_squares = rdd.map(lambda x: x ** 2).reduce(lambda a, b: a + b)
print(sum_of_squares)

55


## 9. TF–IDF

TF–IDF weighs a term by how often it occurs in one document, offset by how many documents
contain it at all. It is the first example here whose PySpark version is not a one-liner, and
it is worth reading for the shape rather than the formula.

**Term frequency** — how often term $t$ occurs in document $d$:

$$TF(t, d) = \text{count of } t \text{ in } d$$

**Inverse document frequency** — how rare $t$ is across the corpus of $N$ documents, where
$df(t)$ is the number of documents containing it:

$$IDF(t) = \log\!\left(\frac{N}{df(t)}\right)$$

**TF–IDF** is their product, $TF(t,d) \times IDF(t)$: high when a term is frequent here and
rare elsewhere.

> This is the plain formulation, and it is the one the code below computes. Implementations
> differ in their smoothing: scikit-learn, for instance, uses
> $\log\!\left(\frac{1+N}{1+df(t)}\right) + 1$, which keeps the weight of a term appearing in
> every document positive rather than zero. Nothing in this notebook depends on the choice.

In [18]:
# Python
documents = ["apple orange apple", "apple lemon", "orange lemon"]
N = len(documents)
idf_dict = Counter()
tf_dict_list = []

for doc in documents:
    tf_dict = Counter(doc.split())
    tf_dict_list.append(tf_dict)
    for word in tf_dict.keys():
        idf_dict[word] += 1

for word in idf_dict.keys():
    idf_dict[word] = math.log(N / idf_dict[word])

tfidf_documents = []
for tf_dict in tf_dict_list:
    tfidf = {}
    for word, count in tf_dict.items():
        tfidf[word] = count * idf_dict[word]
    tfidf_documents.append(tfidf)

print(tfidf_documents)

[{'apple': 0.8109302162163288, 'orange': 0.4054651081081644}, {'apple': 0.4054651081081644, 'lemon': 0.4054651081081644}, {'orange': 0.4054651081081644, 'lemon': 0.4054651081081644}]


In [19]:
# PySpark
documents = ["apple orange apple", "apple lemon", "orange lemon"]
rdd = sc.parallelize(documents)

# TF: one Counter per document, computed independently of every other document.
tf_rdd = rdd.map(lambda doc: Counter(doc.split()))

# IDF: how many documents contain each term.  This is word count over the DISTINCT terms
# of each document, and it is the only step in the whole computation that shuffles.
idf_rdd = (tf_rdd.flatMap(lambda tf: tf.keys())
                 .map(lambda word: (word, 1))
                 .reduceByKey(lambda a, b: a + b))

N = rdd.count()
idf_values = idf_rdd.mapValues(lambda count: math.log(N / count)).collectAsMap()

# TF-IDF: the small IDF table is now a driver dict, shipped to the executors in the closure.
# Chapter 2 shows why sc.broadcast is the right way to send a table of any real size.
tfidf_rdd = tf_rdd.map(lambda tf: {word: tf[word] * idf_values[word] for word in tf})

tfidf = tfidf_rdd.collect()
print(tfidf)

[{'apple': 0.8109302162163288, 'orange': 0.4054651081081644}, {'apple': 0.4054651081081644, 'lemon': 0.4054651081081644}, {'orange': 0.4054651081081644, 'lemon': 0.4054651081081644}]


In [20]:
# The two implementations must agree.  An assertion that runs beats a sentence claiming they do.
for a, b in zip(tfidf_documents, tfidf):
    assert a.keys() == b.keys()
    for k in a:
        assert abs(a[k] - b[k]) < 1e-12, (k, a[k], b[k])
print("the Python and PySpark TF-IDF results agree to within floating-point tolerance")

the Python and PySpark TF-IDF results agree to within floating-point tolerance


## Conclusion

Across all seven algorithms the sequential version does the same three things: it fixes an
**order**, it keeps a **mutable accumulator**, and it holds all the data in **one place**. Each
of those is exactly what a cluster cannot provide, which is why the loop has to go.

The PySpark version replaces them with two properties that a distributed system *can* honour:

* **element-wise** work — `map`, `filter`, `flatMap` — where each record is handled without
  reference to any other, so all of them can be handled at once;
* **associative** combining — `reduce`, `reduceByKey`, `countByValue` — where the system may
  choose the grouping, because the answer does not depend on it.

Section 5 is the reminder that importing `pyspark` guarantees neither. `collect()` followed by
a Python loop has both properties available and uses neither.

**Next.** Chapter 2 takes the RDD apart: what it actually is, why laziness is not an
inconvenience, and where the cost of a distributed computation really goes.